In [ ]:
# @title  Setup e Mount Drive
from google.colab import drive
import os

drive.mount('/content/drive')

# Install libraries (if necessary)
!pip install librosa tqdm scikit-learn numpy

print("\n Setup completed.")

In [ ]:
# @title  Paths Settings
import os

# Change this path to your project folder on Drive
PROJECT_ROOT = '/content/drive/MyDrive/Speach_Emotion_Recognition' # @param {type:"string"}

SOURCE_PATH = os.path.join(PROJECT_ROOT, 'dataset/RawData')
# The SAVE_PATH will be automatically updated based on the split choice

EMOTIONS = {
    '01': 'neutral', '02': 'calm', '03': 'happy', '04': 'sad',
    '05': 'angry', '06': 'fearful', '07': 'disgust', '08': 'surprised'
}

print(f" Source: {SOURCE_PATH}")

In [ ]:
# @title Augmentation & Feature Extraction Functions
import librosa
import numpy as np
import random

def add_noise(data, noise_factor_range=(0.005, 0.02)):
    noise_factor = np.random.uniform(*noise_factor_range)
    noise = np.random.randn(len(data))
    return data + noise_factor * noise

def shift_pitch(data, sr, pitch_factor_range=(-2.5, 2.5)):
    n_steps = np.random.uniform(*pitch_factor_range)
    return librosa.effects.pitch_shift(data, sr=sr, n_steps=n_steps)

def stretch_time(data, rate_range=(0.8, 1.2)):
    rate = np.random.uniform(*rate_range)
    return librosa.effects.time_stretch(y=data, rate=rate)

def shift_time(data, shift_max=0.2, sr=22050):
    shift_amt = int(np.random.uniform(-shift_max, shift_max) * sr)
    return np.roll(data, shift_amt)

def apply_random_augmentation(data, sr):
    augmented_data = data.copy()
    if random.random() < 0.5: augmented_data = stretch_time(augmented_data)
    if random.random() < 0.5: augmented_data = shift_pitch(augmented_data, sr)
    if random.random() < 0.8: augmented_data = add_noise(augmented_data)
    if random.random() < 0.5: augmented_data = shift_time(augmented_data, sr=sr)
    return augmented_data

def get_mel_spectrogram(data, sr=22050, target_length=3.0):
    data, _ = librosa.effects.trim(data, top_db=20)
    target_samples = int(target_length * sr)
    if len(data) > target_samples:
        start = (len(data) - target_samples) // 2
        data = data[start : start + target_samples]
    else:
        data = np.pad(data, (0, max(0, target_samples - len(data))), 'constant')
    
    mel_spec = librosa.feature.melspectrogram(y=data, sr=sr, n_mels=128, n_fft=2048, hop_length=512)
    mel_spec_db = librosa.power_to_db(mel_spec, ref=np.max)
    
    if mel_spec_db.shape[1] > 128:
        mel_spec_db = mel_spec_db[:, :128]
    else:
        mel_spec_db = np.pad(mel_spec_db, ((0, 0), (0, 128 - mel_spec_db.shape[1])), 'constant')
    return mel_spec_db[..., np.newaxis]

In [ ]:
# @title Execution Dataset Preparation
import glob
from tqdm.notebook import tqdm
from sklearn.model_selection import train_test_split

tipo_preparazione = "Actor Split" # @param ["Actor Split", "Random Split", "Pure Dataset"]

# Automatic configuration based on choice
augment = True
split_type = "Actor Split"
folder_name = ""

if tipo_preparazione == "Actor Split":
    folder_name = "augmentedDataset_9K_ActorSplit"
elif tipo_preparazione == "Random Split":
    split_type = "Random Split"
    folder_name = "augmentedDataset_9K_RandomSplit"
else:
    split_type = "Random Split"
    augment = False
    folder_name = "pureDataset_RandomSplit"

save_path = os.path.join(PROJECT_ROOT, 'dataset', folder_name)
os.makedirs(save_path, exist_ok=True)

# 1. File upload
all_files = glob.glob(os.path.join(SOURCE_PATH, "Actor_*", "*.wav"))
print(f" Found {len(all_files)} files. Starting {tipo_preparazione}...")

# 2. Logic Split
all_labels = [EMOTIONS.get(os.path.basename(f).split('-')[2]) for f in all_files]
if split_type == "Actor Split":
    train_files = [f for f in all_files if int(os.path.basename(f).split('-')[6].split('.')[0]) < 21]
    test_files = [f for f in all_files if int(os.path.basename(f).split('-')[6].split('.')[0]) >= 21]
else:
    train_files, test_files = train_test_split(all_files, test_size=0.20, random_state=42, stratify=all_labels)

# 3. Processing
X_train, y_train, X_test, y_test = [], [], [], []

for file_list, is_train in [(train_files, True), (test_files, False)]:
    desc = "Elaborazione Train" if is_train else "Elaborazione Test"
    for file_path in tqdm(file_list, desc=desc):
        label = EMOTIONS.get(os.path.basename(file_path).split('-')[2])
        data, sr = librosa.load(file_path, sr=22050)
        
        # Original
        target_X = X_train if is_train else X_test
        target_y = y_train if is_train else y_test
        target_X.append(get_mel_spectrogram(data, sr=sr))
        target_y.append(label)
        
        # Augmentation (Only Validation if is required)
        if is_train and augment:
            num_aug = 12 if label == 'neutral' else 6
            for _ in range(num_aug):
                aug_data = apply_random_augmentation(data, sr)
                X_train.append(get_mel_spectrogram(aug_data, sr=sr))
                y_train.append(label)

# 4. Save
np.save(os.path.join(save_path, 'X_train.npy'), np.array(X_train))
np.save(os.path.join(save_path, 'y_train.npy'), np.array(y_train))
np.save(os.path.join(save_path, 'X_test.npy'), np.array(X_test))
np.save(os.path.join(save_path, 'y_test.npy'), np.array(y_test))

print(f"\n Completed! Dataset saved in {save_path}")